# Home Credit Default Risk

Кагловский датасет: 7 таблиц, надо предсказать дефолт по заявке. Интересно тут
не моделирование (там всё равно будет бустинг), а сколько сигнала лежит
в истории клиента: бюро, прошлые заявки, график платежей. По голой анкете
получается около 0.75 AUC, вопрос в том, сколько добавит агрегация истории.

Метрика ROC-AUC. В конце ещё прикидываю порог отсечения: AUC сам по себе
ничего не говорит о том, кому выдавать.

In [ ]:
import gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.9)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11})

DATA = "home-credit-default-risk"
SEED, EPS = 42, 1e-6
np.random.seed(SEED)

## Что в данных

In [ ]:
app = pd.read_csv(f"{DATA}/application_train.csv")
print(app.shape, f"дефолтов {app.TARGET.mean():.2%}")

# считаю строки не загружая, тут суммарно под 60 млн
for f in ["bureau", "bureau_balance", "previous_application",
          "POS_CASH_balance", "installments_payments", "credit_card_balance"]:
    n = sum(1 for _ in open(f"{DATA}/{f}.csv")) - 1
    print(f"  {f:24s} {n:>10,}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.5))

app.TARGET.value_counts().sort_index().plot.bar(ax=ax[0], rot=0, width=0.5)
ax[0].set_title(f"классы: {app.TARGET.mean():.1%} дефолтов")

for t, lab in [(0, "вернул"), (1, "дефолт")]:
    sns.kdeplot(app.loc[app.TARGET == t, "EXT_SOURCE_3"].dropna(),
                ax=ax[1], fill=True, alpha=.2, label=lab)
ax[1].legend(); ax[1].set_title("EXT_SOURCE_3")

age = (app.DAYS_BIRTH.abs() / 365.25)
r = app.groupby(pd.qcut(age, 10), observed=True).TARGET.mean()
ax[2].plot(range(len(r)), r.values * 100, "o-")
ax[2].axhline(app.TARGET.mean() * 100, ls="--", c="gray", lw=1)
ax[2].set_title("дефолты по децилям возраста, %")
ax[2].set_xlabel("дециль (моложе → старше)")
plt.tight_layout()

print(app[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].corrwith(app.TARGET).round(3))

EXT_SOURCE это единственное, что вообще коррелирует с таргетом на уровне 0.16-0.18,
у всего остального |corr| < 0.08. Молодые заёмщики рискуют примерно вдвое чаще
старших. Ничего неожиданного, но задаёт ожидания.

In [ ]:
# DAYS_EMPLOYED = 365243 это "1000 лет стажа", т.е. закодированный пропуск
anom = app.DAYS_EMPLOYED == 365243
print(f"сентинелов: {anom.sum():,} ({anom.mean():.1%})")
print(f"дефолт у них {app.TARGET[anom].mean():.2%} против {app.TARGET[~anom].mean():.2%}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(app.DAYS_EMPLOYED, bins=60)
ax[0].set_yscale("log"); ax[0].set_title("DAYS_EMPLOYED как есть")
ax[1].hist(app.DAYS_EMPLOYED[~anom].abs() / 365.25, bins=60)
ax[1].set_title("после выкидывания сентинела, лет")
plt.tight_layout()

del app, age; gc.collect()

У группы с сентинелом риск заметно **ниже** среднего, то есть сам факт этого
значения информативен. Значит меняю на NaN, но оставляю флаг.

## Признаки

Главное тут свернуть отношение «один ко многим» в вектор на клиента.
Делаю тупо: числовые колонки → min/max/mean/sum по SK_ID_CURR, плюс руками
несколько полей, которые точно осмысленны (просрочка, недоплата, утилизация).
Отдельно считаю агрегаты по срезам (активные кредиты, отказы, последний год):
разница между «недавно» и «вообще» это отдельный сигнал.

In [ ]:
def agg(df, prefix, funcs=("min", "max", "mean", "sum")):
    """числовые колонки -> статистики на SK_ID_CURR"""
    num = df.select_dtypes("number").drop(columns=["SK_ID_PREV", "SK_ID_BUREAU"],
                                          errors="ignore")
    g = num.groupby("SK_ID_CURR").agg(funcs)
    g.columns = [f"{prefix}_{a}_{b}".upper() for a, b in g.columns]
    g[f"{prefix}_CNT"] = df.groupby("SK_ID_CURR").size()
    return g.astype(np.float32)


def load(name):
    return pd.read_csv(f"{DATA}/{name}.csv")

In [ ]:
t0 = time.time()

tr, te = load("application_train"), load("application_test")
df = pd.concat([tr, te], ignore_index=True)
df = df[df.CODE_GENDER != "XNA"].reset_index(drop=True)
ntr = int(df.TARGET.notna().sum())
del tr, te

df["EMPLOYED_ANOM"] = (df.DAYS_EMPLOYED == 365243).astype(np.int8)
df["DAYS_EMPLOYED"] = df.DAYS_EMPLOYED.replace(365243, np.nan)
for c in ["DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH"]:
    df[c] = df[c].abs()

# абсолютные суммы почти бесполезны, работают отношения
df["CREDIT_INCOME"] = df.AMT_CREDIT / (df.AMT_INCOME_TOTAL + EPS)
df["ANNUITY_INCOME"] = df.AMT_ANNUITY / (df.AMT_INCOME_TOTAL + EPS)
df["CREDIT_TERM"] = df.AMT_ANNUITY / (df.AMT_CREDIT + EPS)
df["CREDIT_GOODS"] = df.AMT_CREDIT / (df.AMT_GOODS_PRICE + EPS)
df["INCOME_PER_PERSON"] = df.AMT_INCOME_TOTAL / (df.CNT_FAM_MEMBERS + EPS)
df["AGE"] = df.DAYS_BIRTH / 365.25
df["EMPLOYED_RATIO"] = df.DAYS_EMPLOYED / (df.DAYS_BIRTH + EPS)

ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
df["EXT_MEAN"] = ext.mean(1)
df["EXT_MIN"] = ext.min(1)
df["EXT_STD"] = ext.std(1)
df["EXT_NAN"] = ext.isna().sum(1)
df["EXT_X_TERM"] = df.EXT_MEAN * df.CREDIT_TERM
df["NAN_CNT"] = df.isna().sum(1)
del ext

df = pd.get_dummies(df, dummy_na=True, dtype=np.int8)
print(f"application: {df.shape}  {time.time()-t0:.0f}s")

In [ ]:
t0 = time.time()

# bureau_balance -> сначала на SK_ID_BUREAU, потом уже к клиенту
bb = load("bureau_balance")
bb["BAD"] = bb.STATUS.isin(list("12345")).astype(np.int8)
bb = bb.groupby("SK_ID_BUREAU").agg(BB_LEN=("MONTHS_BALANCE", "size"),
                                    BB_OLD=("MONTHS_BALANCE", "min"),
                                    BB_BAD=("BAD", "mean"))

bu = load("bureau").join(bb, on="SK_ID_BUREAU")
bu["DURATION"] = bu.DAYS_CREDIT_ENDDATE - bu.DAYS_CREDIT
bu["DEBT_RATIO"] = bu.AMT_CREDIT_SUM_DEBT / (bu.AMT_CREDIT_SUM + EPS)
bu["OVERDUE"] = (bu.CREDIT_DAY_OVERDUE > 0).astype(np.int8)

buro = agg(bu, "BURO")
buro = buro.join(agg(bu[bu.CREDIT_ACTIVE == "Active"], "BURO_ACT"))
buro = buro.join(agg(bu[bu.DAYS_CREDIT >= -730], "BURO_2Y"))
buro["BURO_ACTIVE_SHARE"] = buro.BURO_ACT_CNT / (buro.BURO_CNT + EPS)
del bb, bu; gc.collect()
print(f"bureau: {buro.shape}  {time.time()-t0:.0f}s")

In [ ]:
t0 = time.time()
pa = load("previous_application")
for c in ["DAYS_FIRST_DUE", "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
    pa[c] = pa[c].replace(365243, np.nan)
pa["APP_CREDIT"] = pa.AMT_APPLICATION / (pa.AMT_CREDIT + EPS)
pa["TERM"] = pa.AMT_ANNUITY / (pa.AMT_CREDIT + EPS)
pa["REFUSED"] = (pa.NAME_CONTRACT_STATUS == "Refused").astype(np.int8)

prev = agg(pa, "PREV")
prev = prev.join(agg(pa[pa.NAME_CONTRACT_STATUS == "Approved"], "PREV_OK"))
prev = prev.join(agg(pa[pa.REFUSED == 1], "PREV_NO"))
prev["PREV_REFUSED_SHARE"] = prev.PREV_NO_CNT.fillna(0) / (prev.PREV_CNT + EPS)

# последняя по времени заявка отдельно, она ближе всего к текущей ситуации
last = (pa.sort_values("DAYS_DECISION").groupby("SK_ID_CURR")
        [["AMT_CREDIT", "DAYS_DECISION", "CNT_PAYMENT", "TERM", "REFUSED"]].last()
        .add_prefix("PREV_LAST_"))
prev = prev.join(last)
del pa, last; gc.collect()
print(f"previous: {prev.shape}  {time.time()-t0:.0f}s")

In [ ]:
t0 = time.time()
ip = load("installments_payments")
ip["PAY_RATIO"] = ip.AMT_PAYMENT / (ip.AMT_INSTALMENT + EPS)
ip["PAY_DIFF"] = ip.AMT_INSTALMENT - ip.AMT_PAYMENT
ip["DPD"] = (ip.DAYS_ENTRY_PAYMENT - ip.DAYS_INSTALMENT).clip(lower=0)
ip["LATE"] = (ip.DPD > 0).astype(np.int8)

ins = agg(ip, "INS")
ins = ins.join(agg(ip[ip.DAYS_INSTALMENT >= -365], "INS_1Y"))
# платил ли в последний год хуже, чем вообще
ins["INS_DPD_TREND"] = ins.INS_1Y_DPD_MEAN - ins.INS_DPD_MEAN
ins["INS_LATE_TREND"] = ins.INS_1Y_LATE_MEAN - ins.INS_LATE_MEAN
del ip; gc.collect()
print(f"installments: {ins.shape}  {time.time()-t0:.0f}s")

In [ ]:
t0 = time.time()
pc = load("POS_CASH_balance")
pc["DPD"] = (pc.SK_DPD > 0).astype(np.int8)
pos = agg(pc, "POS").join(agg(pc[pc.MONTHS_BALANCE >= -12], "POS_1Y"))
del pc; gc.collect()

cc = load("credit_card_balance")
cc["UTIL"] = cc.AMT_BALANCE / (cc.AMT_CREDIT_LIMIT_ACTUAL + EPS)
cc["DPD"] = (cc.SK_DPD > 0).astype(np.int8)
card = agg(cc, "CC").join(agg(cc[cc.MONTHS_BALANCE >= -12], "CC_1Y"))
del cc; gc.collect()
print(f"pos {pos.shape}, cards {card.shape}  {time.time()-t0:.0f}s")

In [ ]:
data = df
for a in [buro, prev, ins, pos, card]:
    data = data.join(a, on="SK_ID_CURR")
del df, buro, prev, ins, pos, card; gc.collect()

# кросс-табличное: долг в бюро и платежи по рассрочкам относительно дохода
data["X_BURO_DEBT_INCOME"] = data.BURO_AMT_CREDIT_SUM_DEBT_SUM / (data.AMT_INCOME_TOTAL + EPS)
data["X_INS_PAY_INCOME"] = data.INS_AMT_PAYMENT_MEAN / (data.AMT_INCOME_TOTAL + EPS)
data["X_PREV_CREDIT"] = data.PREV_AMT_CREDIT_MEAN / (data.AMT_CREDIT + EPS)

data = data.loc[:, ~data.columns.duplicated()]   # на всякий, join мог продублировать
data = data.replace([np.inf, -np.inf], np.nan)
f64 = data.select_dtypes("float64").columns
data[f64] = data[f64].astype(np.float32)

print(f"итого {data.shape[0]:,} x {data.shape[1]}, "
      f"{data.memory_usage(deep=True).sum()/1e9:.1f} ГБ, обучение {ntr:,}")

## Отбор

Половина колонок это мусор: константы, почти пустые, и те, которые бустинг
ни разу не взял для разбиения. Гоняю быстрый LGBM на holdout, беру топ по gain.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

train = data.iloc[:ntr]
y = train.TARGET.astype(np.int8).values

nun, miss = train.nunique(), train.isna().mean()
drop = set(nun[nun <= 1].index) | set(miss[miss > .99].index) | {"SK_ID_CURR", "TARGET"}
cand = [c for c in data.columns if c not in drop]
print(f"живых колонок {len(cand)} из {data.shape[1]}")

Xa = train[cand].to_numpy(np.float32)
Xt_, Xv_, yt_, yv_ = train_test_split(Xa, y, test_size=.2, stratify=y, random_state=SEED)

probe = lgb.LGBMClassifier(n_estimators=700, learning_rate=.05, num_leaves=34,
                           colsample_bytree=.35, subsample=.87, subsample_freq=1,
                           min_child_weight=40, n_jobs=-1, verbose=-1, random_state=SEED)
probe.fit(Xt_, yt_, eval_set=[(Xv_, yv_)], eval_metric="auc",
          callbacks=[lgb.early_stopping(100, verbose=False)])
imp = pd.Series(probe.booster_.feature_importance("gain"), index=cand).sort_values(ascending=False)
print(f"нулевой gain у {(imp == 0).sum()} признаков")

# сколько брать: смотрю глазами на три точки
pos_ = {c: i for i, c in enumerate(cand)}
for K in [200, 500, int((imp > 0).sum())]:
    idx = [pos_[c] for c in imp.head(K).index]
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=.03, num_leaves=34,
                           colsample_bytree=.35, subsample=.87, subsample_freq=1,
                           min_child_weight=40, n_jobs=-1, verbose=-1, random_state=SEED)
    tk = time.time()
    m.fit(Xt_[:, idx], yt_, eval_set=[(Xv_[:, idx], yv_)], eval_metric="auc",
          callbacks=[lgb.early_stopping(150, verbose=False)])
    auc = roc_auc_score(yv_, m.predict_proba(Xv_[:, idx])[:, 1])
    print(f"  K={K:5d}  auc={auc:.5f}  {time.time()-tk:.0f}s")

del Xa, Xt_, Xv_, m, probe; gc.collect()

Дальше 500 прирост уже в пределах шума, а время растёт вдвое. Беру 500.

In [ ]:
FEATS = list(imp.head(500).index)
X = train[FEATS].to_numpy(np.float32)
XT = data.iloc[ntr:][FEATS].to_numpy(np.float32)
ID_TEST = data.iloc[ntr:].SK_ID_CURR.values

# та же выборка, но только сырая анкета, нужна как точка отсчёта
raw = pd.concat([load("application_train"), load("application_test")], ignore_index=True)
raw = raw[raw.CODE_GENDER != "XNA"].reset_index(drop=True)
raw = pd.get_dummies(raw.drop(columns=["SK_ID_CURR", "TARGET"]), dummy_na=True, dtype=np.int8)
XR, XRT = raw[:ntr].to_numpy(np.float32), raw[ntr:].to_numpy(np.float32)
del raw, train, data; gc.collect()

src = pd.Series([c.split("_")[0] if c.split("_")[0] in
                 ("BURO", "PREV", "INS", "POS", "CC", "X") else "app" for c in FEATS])
print(X.shape, "против сырой анкеты", XR.shape)
print(src.value_counts().to_dict())

## Модели

Один и тот же 5-фолдовый стратифицированный CV для всех, метрики считаю по
единому OOF-вектору. NaN не заполняю, бустинги выбирают ветку для пропуска
сами, а тут пропуск сам по себе информативен.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import ks_2samp
import xgboost as xgb
from catboost import CatBoostClassifier

FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y))
oof, test, res = {}, {}, {}


def cv(name, fit, Xf, Xtf):
    t0 = time.time()
    o, t = np.zeros(len(y)), np.zeros(len(Xtf))
    for i, (itr, iva) in enumerate(FOLDS):
        pv, pt = fit(Xf[itr], y[itr], Xf[iva], y[iva], Xtf)
        o[iva] = pv
        t += pt / len(FOLDS)
        print(f"   фолд {i+1}  {roc_auc_score(y[iva], pv):.5f}", flush=True)
    oof[name], test[name] = o, t
    res[name] = dict(auc=roc_auc_score(y, o), ap=average_precision_score(y, o),
                     ks=ks_2samp(o[y == 1], o[y == 0]).statistic,
                     sec=time.time() - t0, n=Xf.shape[1])
    print(f"{name:10s} auc={res[name]['auc']:.5f}  ks={res[name]['ks']:.3f}  "
          f"{res[name]['sec']:.0f}s")

In [ ]:
def logreg(Xt, yt, Xv, yv, Xs):
    p = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=.05, max_iter=400, class_weight="balanced"))
    p.fit(Xt, yt)
    return p.predict_proba(Xv)[:, 1], p.predict_proba(Xs)[:, 1]


cv("LogReg raw", logreg, XR, XRT)

In [ ]:
GAIN = []

def fit_lgb(Xt, yt, Xv, yv, Xs):
    m = lgb.LGBMClassifier(n_estimators=6000, learning_rate=.02, num_leaves=34, max_depth=8,
                           colsample_bytree=.35, subsample=.87, subsample_freq=1,
                           reg_alpha=.04, reg_lambda=.07, min_child_weight=40,
                           min_split_gain=.02, n_jobs=-1, verbose=-1, random_state=SEED)
    m.fit(Xt, yt, eval_set=[(Xv, yv)], eval_metric="auc",
          callbacks=[lgb.early_stopping(200, verbose=False)])
    GAIN.append(m.booster_.feature_importance("gain"))
    return m.predict_proba(Xv)[:, 1], m.predict_proba(Xs)[:, 1]


def fit_xgb(Xt, yt, Xv, yv, Xs):
    m = xgb.XGBClassifier(n_estimators=6000, learning_rate=.02, max_depth=6,
                          min_child_weight=30, subsample=.85, colsample_bytree=.35,
                          reg_alpha=.05, reg_lambda=1., gamma=.05, tree_method="hist",
                          max_bin=64, eval_metric="auc", n_jobs=-1,
                          early_stopping_rounds=200, random_state=SEED)
    m.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
    return m.predict_proba(Xv)[:, 1], m.predict_proba(Xs)[:, 1]


def fit_cat(Xt, yt, Xv, yv, Xs):
    m = CatBoostClassifier(iterations=4000, learning_rate=.04, depth=6, l2_leaf_reg=8,
                           rsm=.35, border_count=64, eval_metric="AUC", random_seed=SEED,
                           od_type="Iter", od_wait=200, verbose=0, thread_count=-1,
                           bootstrap_type="Bernoulli", subsample=.85)
    m.fit(Xt, yt, eval_set=(Xv, yv), use_best_model=True)
    return m.predict_proba(Xv)[:, 1], m.predict_proba(Xs)[:, 1]


cv("LightGBM", fit_lgb, X, XT)   # ~15 мин на моей машине
cv("XGBoost", fit_xgb, X, XT)
cv("CatBoost", fit_cat, X, XT)

In [ ]:
# бленд по рангам, а не по вероятностям: модели откалиброваны по-разному
from scipy.stats import rankdata

B = ["LightGBM", "XGBoost", "CatBoost"]
oof["Blend"] = np.mean([rankdata(oof[m]) / len(y) for m in B], 0)
test["Blend"] = np.mean([rankdata(test[m]) / len(test[m]) for m in B], 0)
res["Blend"] = dict(auc=roc_auc_score(y, oof["Blend"]),
                    ap=average_precision_score(y, oof["Blend"]),
                    ks=ks_2samp(oof["Blend"][y == 1], oof["Blend"][y == 0]).statistic,
                    sec=sum(res[m]["sec"] for m in B), n=X.shape[1])

board = pd.DataFrame(res).T.astype(float).sort_values("auc", ascending=False)
best = board.index[0]
print(pd.DataFrame({m: rankdata(oof[m]) for m in B}).corr("spearman").round(3))
board.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

for m in board.index:
    fpr, tpr, _ = roc_curve(y, oof[m])
    ax[0].plot(fpr, tpr, lw=1.6, label=f"{m} {res[m]['auc']:.4f}")
ax[0].plot([0, 1], [0, 1], "--", c="gray", lw=1)
ax[0].legend(fontsize=8); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[0].set_title("ROC (out-of-fold)")

board.auc.sort_values().plot.barh(ax=ax[1], width=.6)
ax[1].set_xlim(.7, board.auc.max() + .01)
ax[1].set_title("ROC-AUC")
plt.tight_layout()

Инженерия истории даёт примерно +0.04 AUC к сырой анкете, ради этого всё и
затевалось. Бленд трёх бустингов добавляет ещё чуть-чуть: корреляция рангов
0.98-0.99, так что много ждать и не приходилось.

## Что выучила модель

In [ ]:
gain = pd.Series(np.mean(GAIN, 0), index=FEATS).sort_values()

fig, ax = plt.subplots(figsize=(6.5, 6))
(gain.tail(25) / gain.sum() * 100).plot.barh(ax=ax)
ax.set_xlabel("доля gain, %"); ax.set_title("топ-25 признаков (LightGBM)")
plt.tight_layout()

by_src = gain.groupby([c.split("_")[0] if c.split("_")[0] in
                       ("BURO", "PREV", "INS", "POS", "CC", "X") else "app"
                       for c in gain.index]).sum() / gain.sum() * 100
print(by_src.sort_values(ascending=False).round(1))

In [ ]:
import shap

ix = np.random.RandomState(SEED).choice(len(X), 6000, replace=False)
m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=.02, num_leaves=34, max_depth=8,
                       colsample_bytree=.35, subsample=.87, subsample_freq=1,
                       min_child_weight=40, n_jobs=-1, verbose=-1, random_state=SEED)
m.fit(X, y)
sv = shap.TreeExplainer(m).shap_values(X[ix])
if isinstance(sv, list):
    sv = sv[1]

shap.summary_plot(sv, pd.DataFrame(X[ix], columns=FEATS), max_display=20, show=False)
plt.title("SHAP", loc="left")
plt.tight_layout()
plt.show()

## Порог

AUC не отвечает на вопрос «кому выдавать». Нужен порог, а для порога цена
ошибки. Считаю, что пропущенный дефолт стоит 10, а отказ хорошему клиенту 1
(для необеспеченной розницы это скорее консервативно). Дальше один проход по
заявкам, отсортированным по скору.

In [ ]:
C_FN, C_FP = 10., 1.
p = oof[best]
o = np.argsort(p)
ys, ps = y[o], p[o]
n = np.arange(1, len(y) + 1)
bad = np.cumsum(ys)                       # дефолты среди одобренных
rej_good = (y == 0).sum() - (n - bad)     # хорошие, которым отказали
cost = (C_FN * bad + C_FP * rej_good) / len(y)
approve, badrate = n / len(y), bad / n

i = int(np.argmin(cost))
T = ps[i]
base = C_FN * y.sum() / len(y)
print(f"порог {T:.4f}: одобрять {approve[i]:.1%}, дефолтов среди них "
      f"{badrate[i]:.2%} (было {y.mean():.2%}), убыток -{(1-cost[i]/base):.0%}")

msk = approve >= .25
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(approve[msk] * 100, cost[msk])
ax[0].scatter(approve[i] * 100, cost[i], c="crimson", zorder=3)
ax[0].axhline(base, ls="--", c="gray", lw=1)
ax[0].set_xlabel("одобрено, %"); ax[0].set_ylabel("убыток на заявку")
ax[0].set_title(f"цена ошибки {C_FN:.0f}:{C_FP:.0f}")

ax[1].plot(approve[msk] * 100, badrate[msk] * 100)
ax[1].axhline(y.mean() * 100, ls="--", c="gray", lw=1)
ax[1].set_xlabel("одобрено, %"); ax[1].set_ylabel("дефолтов среди одобренных, %")
ax[1].set_title("объём против риска")
plt.tight_layout()

In [ ]:
sub = pd.DataFrame({"SK_ID_CURR": ID_TEST, "TARGET": test[best]})
sub.to_csv("submission.csv", index=False)
print(f"{len(sub):,} строк, модель {best}")
sub.head()

## Итого

* История клиента стоит примерно +0.04 AUC к сырой анкете, это главный вывод.
  Больше половины gain уходит в bureau и installments.
* Сильнее всего работают EXT_SOURCE и производные от них, дальше срок кредита
  и просрочки по прошлым рассрочкам.
* Порог это не 0.5, а функция от соотношения цен ошибок. При 10:1 получается
  одобрять около 80 % заявок и убирать примерно треть дефолтов.

Что не доделано / хочется попробовать:

* target encoding по ORGANIZATION_TYPE и OCCUPATION_TYPE (пока просто one-hot)
* стекинг вместо рангового усреднения, веса подбирать честно, на отдельном фолде
* признаки последовательности платежей (сейчас всё схлопнуто в средние,
  а порядок просрочек наверняка что-то значит)
* гиперпараметры руками, Optuna не гонял